# Building CodAdapt

## Goal

Build CodAdapt from scratch to understand how the algorithm works step by step.

CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

This project rebuilds the core ideas from the simplest possible version and gradually introduces bucketization, coded memory, collisions, and adaptive codes.

---

## Version 4

Combine multiple coded features into a shared memory.

The model will:

- discretize `Age` and `Fare`
- learn codes for their buckets
- combine the codes into a shared memory address
- use the shared memory to make predictions

In [1]:
import numpy as np
import pandas as pd

In [2]:
PATH = "/kaggle/input/competitions/titanic/"

train = pd.read_csv(PATH + "train.csv", index_col="PassengerId")
test  = pd.read_csv(PATH + "test.csv", index_col="PassengerId")

## Data preparation

We use two numerical features: `Age` and `Fare`, with `Survived` as the target.

Missing values are filled using the medians learned from the training data.

In [3]:
target = "Survived"

age_median = train["Age"].median()
fare_median = train["Fare"].median()

age = train["Age"].fillna(age_median).to_numpy()
fare = train["Fare"].fillna(fare_median).to_numpy()
y = train[target].to_numpy()

## Building the model

Each feature is independently divided into four buckets.

In [4]:
age_cuts = np.quantile(age, [0.25, 0.50, 0.75])
fare_cuts = np.quantile(fare, [0.25, 0.50, 0.75])

age_bucket = np.digitize(age, age_cuts)
fare_bucket = np.digitize(fare, fare_cuts)

In [5]:
age_mean = np.array([y[age_bucket == b].mean() for b in range(4)])
fare_mean = np.array([y[fare_bucket == b].mean() for b in range(4)])

age_codes = (age_mean >= y.mean()).astype(int)
fare_codes = (fare_mean >= y.mean()).astype(int)

The codes of both features are combined into one shared memory address.

In [6]:
address = age_codes[age_bucket] + 2 * fare_codes[fare_bucket]

In [7]:
memory = np.zeros(4)

for a in range(4):
    if np.any(address == a):
        memory[a] = y[address == a].mean()

In [8]:
memory

array([0.24590164, 0.25680934, 0.45588235, 0.53650794])

A prediction is obtained by using the code as an address into memory.

In [9]:
preds = memory[address]

In [10]:
preds[:10]

array([0.25680934, 0.53650794, 0.25680934, 0.53650794, 0.25680934,
       0.24590164, 0.53650794, 0.53650794, 0.25680934, 0.53650794])

## Prediction and submission

The same bucket boundaries, codes, and shared memory learned from the training data are reused for the test data.

In [11]:
age_test = test["Age"].fillna(age_median).to_numpy()
fare_test = test["Fare"].fillna(fare_median).to_numpy()

age_bucket_test = np.digitize(age_test, age_cuts)
fare_bucket_test = np.digitize(fare_test, fare_cuts)

address_test = age_codes[age_bucket_test] + 2 * fare_codes[fare_bucket_test]

probabilities = memory[address_test]
preds = (probabilities >= 0.5).astype(int)

In [12]:
submission = pd.DataFrame({
    "PassengerId": test.index,
    "Survived": preds
})

submission.to_csv("submission_v4.csv", index=False)

submission.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


## Notes

- This fourth version combines multiple coded features into a shared memory.
- `Age` and `Fare` are independently discretized into four buckets.
- Each bucket receives a simple adaptive code based on its target mean.
- The feature codes are combined to produce a shared memory address.
- Many original bucket combinations can therefore share the same memory cell.
- The memory stores the average target value for each address.
- The learned bucket boundaries, codes, and memory are reused for the test data.
- The production library uses a more sophisticated adaptive coded memory and shared multi-resolution encoding.
- This notebook is a simplified educational reconstruction of CodAdapt, not the full library implementation.
- CodAdapt is an experimental tabular machine learning library I am currently developing, available on GitHub and PyPI.

Future versions will introduce new components and gradually evolve the architecture.